# Parte 3: Comparación con scikit-learn

**Trabajo Práctico — Matemática III: Redes Neuronales**  
**Dataset:** Water Potability  
**Integrantes:** Micaela Ortiz y Camila Maldonado

---

En esta parte implementamos la misma red neuronal de la Parte 2 usando `scikit-learn`, comparamos los resultados y exploramos arquitecturas alternativas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

## Preprocesamiento

Reproducimos exactamente el mismo preprocesamiento y split que en la Parte 2, con la misma `seed=42`, para que la comparación sea válida.

In [ ]:
# Carga y limpieza de datos (idéntica a Parte 2)
df = pd.read_csv("../data/water_potability.csv")

df["ph"]             = df["ph"].fillna(df["ph"].median())
df["Sulfate"]        = df["Sulfate"].fillna(df["Sulfate"].median())
df["Trihalomethanes"]= df["Trihalomethanes"].fillna(df["Trihalomethanes"].median())

X = df.drop(columns=["Potability"]).values.astype(float)
y = df["Potability"].values

# Normalización Z-score
X = (X - X.mean(axis=0)) / X.std(axis=0)

# Split 70/30 con la misma semilla que la Parte 2
np.random.seed(42)
indices = np.random.permutation(len(X))
corte   = int(len(X) * 0.7)

X_train = X[indices[:corte]]
X_test  = X[indices[corte:]]
y_train = y[indices[:corte]]
y_test  = y[indices[corte:]]

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)

## (a) Implementación con la misma arquitectura

La red neuronal de la Parte 2 tiene:
- **Entrada:** 9 neuronas (una por cada variable físico-química)
- **Capa oculta:** 8 neuronas con activación **ReLU**
- **Salida:** 1 neurona con activación **sigmoide**
- **Optimizador:** SGD con learning rate = 0.01
- **Épocas:** 1000

Replicamos esto con `MLPClassifier`:

In [ ]:
# Modelo base: misma arquitectura que la Parte 2
# hidden_layer_sizes=(8,) → 1 capa oculta con 8 neuronas
# activation='relu'       → misma función de activación
# solver='sgd'            → mismo optimizador
# learning_rate_init=0.01 → misma tasa de aprendizaje
# La salida usa sigmoide internamente para clasificación binaria

mlp_base = MLPClassifier(
    hidden_layer_sizes=(8,),
    activation='relu',
    solver='sgd',
    learning_rate_init=0.01,
    max_iter=1000,
    random_state=42,
    n_iter_no_change=1000   # desactivar early stopping para comparar fairamente
)

inicio = time.time()
mlp_base.fit(X_train, y_train)
tiempo_base = time.time() - inicio

print(f"Entrenamiento completado en {tiempo_base:.2f}s")
print(f"Iteraciones realizadas: {mlp_base.n_iter_}")

In [ ]:
# Métricas del modelo base
y_pred_train_base = mlp_base.predict(X_train)
y_pred_test_base  = mlp_base.predict(X_test)
y_prob_test_base  = mlp_base.predict_proba(X_test)[:, 1]

acc_train_base = accuracy_score(y_train, y_pred_train_base)
acc_test_base  = accuracy_score(y_test,  y_pred_test_base)
f1_base        = f1_score(y_test, y_pred_test_base)
auc_base       = roc_auc_score(y_test, y_prob_test_base)

print("=" * 45)
print("  Modelo sklearn (8 neuronas, ReLU, SGD)")
print("=" * 45)
print(f"  Accuracy train : {acc_train_base:.4f}")
print(f"  Accuracy test  : {acc_test_base:.4f}")
print(f"  F1-score test  : {f1_base:.4f}")
print(f"  AUC-ROC test   : {auc_base:.4f}")
print()
print("Reporte completo:")
print(classification_report(y_test, y_pred_test_base, target_names=["No potable", "Potable"]))

In [ ]:
# Curva de pérdida del modelo base
plt.figure(figsize=(9, 4))
plt.plot(mlp_base.loss_curve_, color='tab:blue')
plt.title('Curva de pérdida — sklearn (8 neuronas, ReLU, SGD)')
plt.xlabel('Época')
plt.ylabel('Pérdida (cross-entropy)')
plt.grid()
plt.tight_layout()
plt.show()

## (b) Comparación con la implementación en numpy

A continuación comparamos los resultados obtenidos en la Parte 2 con los obtenidos aquí. Los valores de numpy provienen de los resultados registrados en el notebook de la Parte 2.

In [ ]:
# Resultados del modelo numpy (Parte 2) — copiados del notebook anterior
# Accuracy train final: 0.6986 | Accuracy test final: 0.6633
# (modelo con 1000 epochs, SGD, lr=0.01, 8 neuronas, seed=42)

resultados = {
    'Implementación': ['numpy (Parte 2)', 'sklearn (Parte 3)'],
    'Acc. train':     [0.6986, acc_train_base],
    'Acc. test':      [0.6633, acc_test_base],
    'F1-score test':  ['—',    f"{f1_base:.4f}"],
    'AUC-ROC test':   ['—',    f"{auc_base:.4f}"],
}

df_comparacion = pd.DataFrame(resultados)
print(df_comparacion.to_string(index=False))

In [ ]:
# Gráfico comparativo de accuracy
# Numpy: valores finales por epoch (cargados manualmente)
acc_numpy_train_final = 0.6986
acc_numpy_test_final  = 0.6633

labels    = ['numpy\n(Parte 2)', 'sklearn\n(Parte 3)']
acc_train = [acc_numpy_train_final, acc_train_base]
acc_test  = [acc_numpy_test_final,  acc_test_base]

x = np.arange(len(labels))
ancho = 0.35

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(x - ancho/2, acc_train, ancho, label='Train', color='tab:blue')
ax.bar(x + ancho/2, acc_test,  ancho, label='Test',  color='tab:red')

for i, (vt, ve) in enumerate(zip(acc_train, acc_test)):
    ax.text(i - ancho/2, vt + 0.005, f'{vt:.4f}', ha='center', fontsize=10)
    ax.text(i + ancho/2, ve + 0.005, f'{ve:.4f}', ha='center', fontsize=10)

ax.set_ylabel('Accuracy')
ax.set_title('Comparación numpy vs sklearn — misma arquitectura')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 0.85)
ax.legend()
ax.grid(axis='y')
plt.tight_layout()
plt.show()

### Análisis de la comparación

Ambas implementaciones producen resultados muy similares, lo que confirma que la red neuronal implementada manualmente en numpy es correcta.

Las pequeñas diferencias que puedan observarse se deben a:
1. **Orden de actualización de pesos:** numpy actualiza de a un ejemplo por vez (SGD puro); sklearn puede diferir en cómo internamente aplica los gradientes.
2. **Inicialización interna:** aunque ambos usan `seed=42`, sklearn aplica su propia estrategia de inicialización (Xavier/Glorot) que difiere de la `randn * 0.01` usada en numpy.
3. **Parada por convergencia:** sklearn puede detenerse antes si detecta estabilización.

**Ventajas de sklearn sobre numpy:**
- El código es mucho más conciso (3 líneas vs decenas de líneas).
- Incluye soporte nativo para `predict_proba`, `early_stopping`, `validation_fraction`.
- Optimizado internamente en C, por lo que es más rápido.
- Permite explorar arquitecturas fácilmente cambiando `hidden_layer_sizes`.

**Ventaja de numpy:**
- La implementación manual permite entender exactamente qué ocurre en cada paso del entrenamiento (forward pass, backpropagation, actualización de pesos).

## (c) Exploración de otras arquitecturas

Aprovechamos la facilidad de sklearn para probar distintas configuraciones de la red. Evaluamos variando:
- **Número de neuronas** en la capa oculta
- **Número de capas ocultas**
- **Función de activación** (ReLU vs tanh)
- **Optimizador** (SGD vs Adam)

Para esta exploración usamos Adam con más iteraciones, que es el optimizador estándar en la práctica.

In [ ]:
arquitecturas = [
    {"nombre": "Base: (8)  ReLU + SGD",    "capas": (8,),      "act": "relu",  "solver": "sgd",  "lr": 0.01},
    {"nombre": "(16)  ReLU + Adam",         "capas": (16,),     "act": "relu",  "solver": "adam", "lr": 0.001},
    {"nombre": "(32)  ReLU + Adam",         "capas": (32,),     "act": "relu",  "solver": "adam", "lr": 0.001},
    {"nombre": "(16, 8)  ReLU + Adam",      "capas": (16, 8),   "act": "relu",  "solver": "adam", "lr": 0.001},
    {"nombre": "(32, 16)  ReLU + Adam",     "capas": (32, 16),  "act": "relu",  "solver": "adam", "lr": 0.001},
    {"nombre": "(32, 16, 8) ReLU + Adam",   "capas": (32,16,8), "act": "relu",  "solver": "adam", "lr": 0.001},
    {"nombre": "(16, 8)  tanh + Adam",      "capas": (16, 8),   "act": "tanh",  "solver": "adam", "lr": 0.001},
]

resultados_arq = []

for arq in arquitecturas:
    modelo = MLPClassifier(
        hidden_layer_sizes=arq["capas"],
        activation=arq["act"],
        solver=arq["solver"],
        learning_rate_init=arq["lr"],
        max_iter=2000,
        random_state=42,
    )
    t0 = time.time()
    modelo.fit(X_train, y_train)
    dt = time.time() - t0

    at = accuracy_score(y_train, modelo.predict(X_train))
    ae = accuracy_score(y_test,  modelo.predict(X_test))
    f1 = f1_score(y_test, modelo.predict(X_test))
    auc= roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1])

    resultados_arq.append({
        "Arquitectura": arq["nombre"],
        "Acc. train":   round(at, 4),
        "Acc. test":    round(ae, 4),
        "F1-score":     round(f1, 4),
        "AUC-ROC":      round(auc,4),
        "Tiempo (s)":   round(dt, 2),
        "_modelo":      modelo,       # guardamos para graficar
    })

df_arq = pd.DataFrame(resultados_arq).drop(columns=["_modelo"])
print(df_arq.to_string(index=False))

In [ ]:
# Gráfico de barras comparativo por arquitectura
nombres = [r["Arquitectura"] for r in resultados_arq]
acc_tests = [r["Acc. test"] for r in resultados_arq]
f1_scores = [r["F1-score"]  for r in resultados_arq]
aucs      = [r["AUC-ROC"]   for r in resultados_arq]

x = np.arange(len(nombres))
ancho = 0.28

fig, ax = plt.subplots(figsize=(13, 5))
b1 = ax.bar(x - ancho,     acc_tests, ancho, label='Acc. test', color='tab:blue')
b2 = ax.bar(x,             f1_scores, ancho, label='F1-score',  color='tab:orange')
b3 = ax.bar(x + ancho,     aucs,      ancho, label='AUC-ROC',   color='tab:green')

ax.set_ylabel('Valor')
ax.set_title('Comparación de arquitecturas')
ax.set_xticks(x)
ax.set_xticklabels(nombres, rotation=30, ha='right', fontsize=8)
ax.set_ylim(0, 0.85)
ax.legend()
ax.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Curvas de pérdida de todos los modelos
fig, ax = plt.subplots(figsize=(10, 5))

colores = plt.cm.tab10.colors
for i, (arq, res) in enumerate(zip(arquitecturas, resultados_arq)):
    modelo = [r["_modelo"] for r in resultados_arq if r["Arquitectura"] == res["Arquitectura"]]
    # recuperamos el modelo del loop anterior
    pass

# Re-entrenamos guardando los modelos para las curvas
for i, arq in enumerate(arquitecturas):
    m = MLPClassifier(
        hidden_layer_sizes=arq["capas"], activation=arq["act"],
        solver=arq["solver"], learning_rate_init=arq["lr"],
        max_iter=2000, random_state=42,
    )
    m.fit(X_train, y_train)
    ax.plot(m.loss_curve_, label=arq["nombre"], color=colores[i], alpha=0.85)

ax.set_xlabel('Época')
ax.set_ylabel('Pérdida')
ax.set_title('Curvas de pérdida por arquitectura')
ax.legend(fontsize=7, loc='upper right')
ax.grid()
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de confusión del mejor modelo (32 neuronas, ReLU, Adam)
mejor_modelo = MLPClassifier(
    hidden_layer_sizes=(32,), activation='relu', solver='adam',
    learning_rate_init=0.001, max_iter=2000, random_state=42
)
mejor_modelo.fit(X_train, y_train)
y_pred_mejor = mejor_modelo.predict(X_test)

cm = confusion_matrix(y_test, y_pred_mejor)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No potable", "Potable"])
disp.plot(cmap='Blues')
plt.title('Matriz de confusión — mejor modelo: (32) ReLU + Adam')
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred_mejor, target_names=["No potable", "Potable"]))

### Análisis de arquitecturas

De los experimentos realizados se pueden extraer las siguientes conclusiones:

**1. Aumentar neuronas ayuda moderadamente:**  
Pasar de 8 a 32 neuronas en la capa oculta mejora ligeramente el accuracy y el F1-score. La red con mayor capacidad puede capturar mejor las interacciones no lineales entre las variables físico-químicas.

**2. Agregar capas ocultas no mejora:**  
Las arquitecturas con 2 y 3 capas ocultas obtienen resultados similares o peores que la red de una sola capa. Esto es consistente con la naturaleza del dataset: las relaciones entre variables no son lo suficientemente complejas como para requerir representaciones jerárquicas profundas.

**3. Adam supera a SGD:**  
El optimizador Adam converge más rápido y de forma más estable que SGD puro, lo que se refleja en curvas de pérdida más suaves y resultados ligeramente mejores.

**4. ReLU vs tanh:**  
Ambas funciones de activación producen resultados comparables en este dataset. ReLU es preferida en la práctica por su simplicidad computacional y menor riesgo de gradientes que desaparecen.

**5. El rendimiento general es moderado:**  
Ninguna arquitectura supera significativamente el 68% de accuracy. Esto refleja la conclusión del análisis exploratorio de la Parte 1: las variables físico-químicas tienen muy baja correlación individual con la potabilidad, lo que hace que el problema sea intrínsecamente difícil para cualquier modelo.

**Modelo seleccionado:** arquitectura `(32,)` con activación ReLU y optimizador Adam, por su mejor balance entre rendimiento, velocidad de convergencia y simplicidad.